<a href="https://colab.research.google.com/github/Kure18/DSN_Mentorship/blob/main/Electric_Vehicle_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importing our data

In [83]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error,mean_absolute_error

Importing dataset for our base line model with 5000 rows (EV_trimmed)

In [84]:
EV_trimmed = pd.read_csv('/content/drive/MyDrive/MentorShip/Electric_Vehicle_Population_Size_History_By_County.csv',nrows=5000)

Importing dataset for our main model with 34671 rows (EV)

In [85]:
EV = pd.read_csv('/content/drive/MyDrive/MentorShip/Electric_Vehicle_Population_Size_History_By_County.csv')

Performing Exploratory Data Analysis (EDA)

In [86]:
EV_trimmed['Date'].fillna(EV_trimmed['Date'].mode()[0],inplace=True)
EV_trimmed['County'].fillna(EV_trimmed['County'].mode()[0],inplace=True)
EV_trimmed['State'].fillna(EV_trimmed['State'].mode()[0],inplace=True)

/tmp/ipykernel_1190/3381024864.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  EV_trimmed['Date'].fillna(EV_trimmed['Date'].mode()[0],inplace=True)
/tmp/ipykernel_1190/3381024864.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, 

Using LabelEncoder to convert object columns into binary codes

In [87]:
from sklearn.preprocessing import LabelEncoder
le_date = LabelEncoder()
le_county = LabelEncoder()
le_state = LabelEncoder()
le_vehicle = LabelEncoder()

In [88]:
EV_trimmed['Date'] = le_date.fit_transform(EV_trimmed['Date'])
EV_trimmed['County'] = le_county.fit_transform(EV_trimmed['County'])
EV_trimmed['State'] = le_state.fit_transform(EV_trimmed['State'])
EV_trimmed['Vehicle Primary Use'] = le_vehicle.fit_transform(EV_trimmed['Vehicle Primary Use'])

Clarifying that there are no missing values in columns

In [89]:
EV_trimmed.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 10 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Date                                      5000 non-null   int64  
 1   County                                    5000 non-null   int64  
 2   State                                     5000 non-null   int64  
 3   Vehicle Primary Use                       5000 non-null   int64  
 4   Battery Electric Vehicles (BEVs)          5000 non-null   int64  
 5   Plug-In Hybrid Electric Vehicles (PHEVs)  5000 non-null   int64  
 6   Electric Vehicle (EV) Total               5000 non-null   int64  
 7   Non-Electric Vehicle Total                5000 non-null   int64  
 8   Total Vehicles                            5000 non-null   int64  
 9   Percent Electric Vehicles                 5000 non-null   float64
dtypes: float64(1), int64(9)
memory usage

Creating logical columns for county's that have percentage of vehicles that are using the mean value of the column which is 5.69

In [90]:
EV_trimmed['Percent Electric Vehicles'].mean()

np.float64(5.547474)

In [91]:
EV_trimmed["Success Threshold"] = EV_trimmed["Percent Electric Vehicles"]>5.54

In [92]:
EV_trimmed['Success Threshold']

,Success Threshold
0,False
1,False
2,True
3,False
4,False
...,...
4995,False
4996,False
4997,False
4998,True


In [93]:
EV_trimmed.head()

,Date,County,State,Vehicle Primary Use,Battery Electric Vehicles (BEVs),Plug-In Hybrid Electric Vehicles (PHEVs),Electric Vehicle (EV) Total,Non-Electric Vehicle Total,Total Vehicles,Percent Electric Vehicles,Success Threshold
0,44,281,25,0,3,0,3,130,133,2.26,False
1,57,239,20,0,1,0,1,40,41,2.44,False
2,112,210,45,0,1,0,1,10,11,9.09,True
3,60,202,3,0,8,9,17,1775,1792,0.95,False
4,25,288,10,0,0,2,2,83,85,2.35,False


In [94]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score

In [95]:
from sklearn.preprocessing import LabelEncoder
le_date = LabelEncoder()
le_state = LabelEncoder()
le_county = LabelEncoder()
le_vehicle = LabelEncoder()
le_success = LabelEncoder()

In [96]:
EV_trimmed["date"] = le_date.fit_transform(EV_trimmed['Date'])
EV_trimmed["county"] = le_county.fit_transform(EV_trimmed['County'])
EV_trimmed["state"] = le_state.fit_transform(EV_trimmed['State'])
EV_trimmed["vehicle"] = le_vehicle.fit_transform(EV_trimmed['Vehicle Primary Use'])
EV_trimmed["success"] = le_success.fit_transform(EV_trimmed['Success Threshold'])

Splitting the data into independent variable X and dependable variable y

In [97]:
X = EV_trimmed.drop(columns=['Success Threshold', 'Date', 'County', 'State', 'Vehicle Primary Use', 'Percent Electric Vehicles', 'success'])
y = EV_trimmed['Success Threshold']

In [98]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [99]:
X.shape

(5000, 9)

In [100]:
y.shape

(5000,)

Introducing our model that is Logistic Regression

In [101]:
logistic_model = LogisticRegression()
logistic_model.fit(X_train,y_train)
logistic_predictions = logistic_model.predict(X_test)

In [102]:
logistic_predictions

array([ True, False, False, False, False, False,  True, False, False,
       False, False, False,  True, False, False, False,  True, False,
        True, False,  True, False, False, False, False, False, False,
        True, False, False,  True, False, False, False,  True, False,
        True, False, False, False,  True,  True, False,  True,  True,
       False, False,  True, False, False, False, False,  True, False,
       False, False,  True, False,  True,  True, False, False, False,
       False,  True, False, False, False, False, False, False,  True,
       False,  True,  True,  True,  True, False,  True, False,  True,
       False, False,  True, False, False, False,  True, False, False,
        True, False, False,  True, False, False, False, False, False,
       False, False, False, False, False, False,  True, False,  True,
       False, False, False, False, False,  True, False, False,  True,
       False,  True,  True, False,  True,  True, False, False, False,
       False, False,

Introducing Decision Tree Classifier

In [103]:
decision_tree_model = DecisionTreeClassifier()
decision_tree_model.fit(X_train,y_train)
decision_tree_predictions = decision_tree_model.predict(X_test)

In [104]:
metrics = {
    "Accuracy" : accuracy_score,
    "Precision" : precision_score,
    "Recall" : recall_score,
    "F1 Score" : f1_score
}

results = {}
for name,func in metrics.items():
  logistic_score = func(y_test,logistic_predictions)
  decision_tree_score = func(y_test,decision_tree_predictions)

  results[name] = {
      "Logistic Regression" : logistic_score,
      "Decision Tree" : decision_tree_score
  }

In [105]:
results

{'Accuracy': {'Logistic Regression': 0.998, 'Decision Tree': 0.997},
 'Precision': {'Logistic Regression': 0.9959016393442623,
  'Decision Tree': 0.9958847736625515},
 'Recall': {'Logistic Regression': 0.9959016393442623,
  'Decision Tree': 0.9918032786885246},
 'F1 Score': {'Logistic Regression': 0.9959016393442623,
  'Decision Tree': 0.9938398357289527}}

Finding the mean for the entire dataset EV we have

In [106]:
EV['Percent Electric Vehicles'].mean()

np.float64(5.698456058377318)

In [107]:
EV["Success Threshold"] = EV["Percent Electric Vehicles"]>5.69

In [108]:
EV['Success Threshold']

,Success Threshold
0,False
1,False
2,True
3,False
4,False
...,...
34666,False
34667,True
34668,False
34669,False


In [109]:
EV.head()

,Date,County,State,Vehicle Primary Use,Battery Electric Vehicles (BEVs),Plug-In Hybrid Electric Vehicles (PHEVs),Electric Vehicle (EV) Total,Non-Electric Vehicle Total,Total Vehicles,Percent Electric Vehicles,Success Threshold
0,January 31 2023,Pulaski,MO,Passenger,3,0,3,130,133,2.26,False
1,June 30 2017,Norfolk,MA,Passenger,1,0,1,40,41,2.44,False
2,September 30 2025,Medina,TX,Passenger,1,0,1,10,11,9.09,True
3,June 30 2020,Maricopa,AZ,Passenger,8,9,17,1775,1792,0.95,False
4,December 31 2023,Richmond,GA,Passenger,0,2,2,83,85,2.35,False


In [110]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score

In [111]:
from sklearn.preprocessing import LabelEncoder
le_date = LabelEncoder()
le_state = LabelEncoder()
le_county = LabelEncoder()
le_vehicle = LabelEncoder()
le_success = LabelEncoder()

In [112]:
EV["date"] = le_date.fit_transform(EV['Date'])
EV["county"] = le_county.fit_transform(EV['County'])
EV["state"] = le_state.fit_transform(EV['State'])
EV["vehicle"] = le_vehicle.fit_transform(EV['Vehicle Primary Use'])
EV["success"] = le_success.fit_transform(EV['Success Threshold'])

In [113]:
X = EV.drop(columns=['Success Threshold', 'Date', 'County', 'State', 'Vehicle Primary Use', 'Percent Electric Vehicles', 'success'])
y = EV['Success Threshold']

In [114]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [123]:
EV.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34671 entries, 0 to 34670
Data columns (total 16 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Date                                      34671 non-null  object 
 1   County                                    34558 non-null  object 
 2   State                                     34558 non-null  object 
 3   Vehicle Primary Use                       34671 non-null  object 
 4   Battery Electric Vehicles (BEVs)          34671 non-null  int64  
 5   Plug-In Hybrid Electric Vehicles (PHEVs)  34671 non-null  int64  
 6   Electric Vehicle (EV) Total               34671 non-null  int64  
 7   Non-Electric Vehicle Total                34671 non-null  int64  
 8   Total Vehicles                            34671 non-null  int64  
 9   Percent Electric Vehicles                 34671 non-null  float64
 10  Success Threshold                 

In [115]:
X.shape

(34671, 9)

In [116]:
y.shape

(34671,)

In [117]:
logistic_model = LogisticRegression()
logistic_model.fit(X_train,y_train)
logistic_predictions = logistic_model.predict(X_test)

In [118]:
logistic_predictions

array([ True, False, False, ..., False, False, False])

In [119]:
decision_tree_model = DecisionTreeClassifier()
decision_tree_model.fit(X_train,y_train)
decision_tree_predictions = decision_tree_model.predict(X_test)

In [120]:
metrics = {
    "Accuracy" : accuracy_score,
    "Precision" : precision_score,
    "Recall" : recall_score,
    "F1 Score" : f1_score
}

results = {}
for name,func in metrics.items():
  logistic_score = func(y_test,logistic_predictions)
  decision_tree_score = func(y_test,decision_tree_predictions)

  results[name] = {
      "Logistic Regression" : logistic_score,
      "Decision Tree" : decision_tree_score
  }

In [121]:
results

{'Accuracy': {'Logistic Regression': 0.9995674116798846,
  'Decision Tree': 0.9991348233597693},
 'Precision': {'Logistic Regression': 1.0,
  'Decision Tree': 0.9986568166554735},
 'Recall': {'Logistic Regression': 0.9979879275653923,
  'Decision Tree': 0.9973172367538564},
 'F1 Score': {'Logistic Regression': 0.998992950654582,
  'Decision Tree': 0.9979865771812081}}